In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier
import lightgbm as lgb
from catboost import CatBoostRegressor, Pool
from sklearn.svm import SVR, SVC
from sklearn.decomposition import PCA
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)


# Show all columns when printing
pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv("src/data/training/raw/merged_training_data.csv")

In [ ]:
df['mix__sd_od_history_to_kyc_ratio'] = df['od__history_length_days'] / df['sd__age_since_first_kyc']

In [ ]:
ordinal_features = [
    "cb__worst_score",
    "sd__membership_product"
]

ordinal_encoder = OrdinalEncoder(
    categories=[
        ["A", "B", "C", "D", "E", "F", "G", "M", "N", "O", "P", np.nan],           # cb__worst_score
        ["PERSONAL_FLEX", "PERSONAL_STANDARD", "PERSONAL_SMART"
         , "PERSONAL_YOU", "PERSONAL_METAL", "BUSINESS_STANDARD"
         , "BUSINESS_SMART", "BUSINESS_YOU", "BUSINESS_METAL", np.nan]        # sd__membership_product
    ],
    encoded_missing_value=np.nan
)

df[ordinal_features] = ordinal_encoder.fit_transform(
    df[ordinal_features]
)

In [ ]:
reg_feats = [
    'ob__open_limit_ref',
 'ob__avg_util_3m',
 'ob__od_enabled_zero_util_days_6m',
 'sd__age',
 'cb__worst_score',
 'ob__max_util_change_3m',
 'ab__inflow_spike_count_6m',
 'ul__is_expat',
 'ab__avg_bal_3m',
 'ob__max_util_3m',
 'sd__max_daily_logins_6m',
 'ab__days_down_6m',
 'ob__max_to_avg_util_3m_ratio',
 'ab__std_util_3m',
 'ob__std_util_3m']

In [ ]:
y_reg = df['t__ccf']

X = df.copy()

In [ ]:
dates = pd.to_datetime(X["reference_date"])

half = np.where(dates.dt.month <= 6, "H1", "H2")
X["half_year_group"] = dates.dt.year.astype(str) + "-" + half

In [ ]:
X_train, X_test, y_train_reg, y_test_reg = train_test_split(X, y_reg, test_size=0.2, random_state=42, stratify=X["half_year_group"])

In [ ]:
X_train_reg = X_train[reg_feats]
X_test_reg = X_test[reg_feats]

In [ ]:
# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_reg)
X_test_scaled = scaler.transform(X_test_reg)

In [ ]:
lin_reg = LinearRegression().fit(X_train_scaled, y_train_reg)
y_reg_pred = lin_reg.predict(X_test_scaled)

In [ ]:
# Side by side
result = np.column_stack((y_reg_pred, y_test_reg))

In [ ]:
result

In [ ]:
mse = mean_squared_error(y_reg_pred, y_test_reg)
print("MSE:", mse)

In [ ]:
plt.figure(figsize=(7,7))
plt.scatter(y_test_reg, y_reg_pred, alpha=0.6)
plt.plot([y_test_reg.min(), y_test_reg.max()],
         [y_test_reg.min(), y_test_reg.max()],
         color='red', linewidth=2, linestyle='--')  # y=x reference line
plt.xlabel("True Values")
plt.ylabel("Predicted Values")
plt.title("Predicted vs True Values")
plt.grid(True)
plt.show()

In [ ]:
dec_tree_reg = DecisionTreeRegressor(random_state=42).fit(X_train_reg, y_train_reg)
y_reg_pred = dec_tree_reg.predict(X_test_reg)

In [ ]:
# Side by side
result = np.column_stack((y_reg_pred, y_test_reg))

In [ ]:
result

In [ ]:
mse = mean_squared_error(y_test_reg, y_reg_pred)
print("MSE:", mse)

In [ ]:
dt = DecisionTreeRegressor(random_state=42)
param_grid = {
    'max_depth': [None, 3, 5, 7, 10, 15],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5, 10],
    'max_features': [None, 'sqrt', 'log2']
}

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=dt,
                           param_grid=param_grid,
                           cv=5,  # 5-fold cross-validation
                           scoring='neg_mean_squared_error',
                           n_jobs=-1)

# Fit Grid Search
grid_search.fit(X_train_reg, y_train_reg)


# Best hyperparameters
print("Best Hyperparameters:", grid_search.best_params_)

In [ ]:
# Evaluate on test set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_reg)
mse = mean_squared_error(y_test_reg, y_pred)
rmse = np.sqrt(mse)
print("Test MSE:", mse)


In [ ]:
# Feature importance
importance_dt = pd.Series(dec_tree_reg.feature_importances_, index=X_train_reg.columns).sort_values(ascending=False)
print("Decision Tree Feature Importance:\n", importance_dt)

# Plot
importance_dt.plot(kind='bar', title='Decision Tree Feature Importance')
plt.show()

In [ ]:
rdm_for_reg = RandomForestRegressor(random_state=42, n_estimators=1000).fit(X_train_reg, y_train_reg)
y_reg_pred = rdm_for_reg.predict(X_test_reg)

In [ ]:
y_reg_pred = rdm_for_reg.predict(X_test_reg)

In [ ]:
rf = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

In [ ]:
param_dist = {
    "n_estimators": [300, 500, 800, 1000],
    "max_depth": [None, 5, 10, 20, 30],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_features": ["sqrt", "log2", 0.5, 0.8],
    "bootstrap": [True, False]
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

rf_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=50,               # increase to 100 if time allows
    cv=5,
    scoring="neg_root_mean_squared_error",  # or "r2"
    n_jobs=-1,
    random_state=42,
    verbose=2
)

rf_search.fit(X_train_reg, y_train_reg)

In [ ]:
best_rf = rf_search.best_estimator_

print("Best parameters:")
print(rf_search.best_params_)

print("Best CV score:")
print(-rf_search.best_score_)

In [ ]:
y_reg_pred = best_rf.predict(X_test_reg)

In [ ]:
# Side by side
result = np.column_stack((y_reg_pred, y_test_reg))

In [ ]:
result

In [ ]:
mse = mean_squared_error(y_test_reg, y_reg_pred)
print("MSE:", mse)

In [ ]:
xgb_reg = XGBRegressor(objective='reg:tweedie',  # correct string for Tweedie in XGBoost >=1.6
    tweedie_variance_power=1,
    n_estimators=500,
    learning_rate=0.01,
    max_depth=5,
    subsample=0.7,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1).fit(X_train_scaled, y_train_reg)
y_reg_pred = xgb_reg.predict(X_test_scaled)

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.9,
    colsample_bytree=0.9,
    min_child_weight=5,
    reg_lambda=1.0,
    objective="reg:squarederror",
    n_jobs=-1
)

xgb.fit(X_train_scaled, y_train_reg)
y_reg_pred = xgb.predict(X_test_scaled)

In [ ]:
from catboost import CatBoostRegressor

cat_reg = CatBoostRegressor(
    loss_function='Tweedie:variance_power=1.1',  # Tweedie with variance_power=1
    iterations=1000,
    learning_rate=0.01,
    depth=5,
    subsample=0.7,
    colsample_bylevel=0.9,  # similar to colsample_bytree
    random_seed=42,
    thread_count=-1,
    verbose=100  # prints progress
).fit(X_train_reg, y_train_reg)

y_reg_pred = cat_reg.predict(X_test_reg)


In [ ]:
import lightgbm as lgb

lgb_reg = lgb.LGBMRegressor(
    objective='tweedie',
    tweedie_variance_power=1,
    n_estimators=500,
    learning_rate=0.01,
    max_depth=5,
    subsample=0.7,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1
).fit(X_train_scaled, y_train_reg)

y_reg_pred = lgb_reg.predict(X_test_scaled)

In [ ]:
# Side by side
result = np.column_stack((y_reg_pred, y_test_reg))

In [ ]:
result[:10]

In [ ]:
X_test.iloc[7]

In [ ]:
mse = mean_squared_error(y_test_reg, y_reg_pred)
print("MSE:", mse)

In [ ]:
plt.figure(figsize=(7,7))
plt.scatter(y_test_reg, y_reg_pred, alpha=0.6)
plt.plot([y_test_reg.min(), y_test_reg.max()],
         [y_test_reg.min(), y_test_reg.max()],
         color='red', linewidth=2, linestyle='--')  # y=x reference line
plt.xlabel("True Values")
plt.ylabel("Predicted Values")
plt.title("Predicted vs True Values")
plt.grid(True)
plt.show()

In [ ]:
test = pd.concat([X_test.reset_index(drop=True), 
                    pd.Series(y_reg_pred, name='predicted_ccf'),
                    pd.Series(y_test_reg.reset_index(drop=True), name='realized_ccf')],
                   axis=1)

In [ ]:
test[((test['predicted_ccf']>0.6) & (test['realized_ccf']<0.25))]

In [ ]:
param_grid = {
    'n_estimators': [500, 1000],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.7, 0.9, 1],
    'colsample_bytree': [0.7, 0.9, 1]
}

xgb = XGBRegressor(
    objective='reg:tweedie',
    tweedie_variance_power=1,
    random_state=42,
    n_jobs=-1
)

grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=3,
    verbose=2
)

grid_search.fit(X_train_scaled, y_train_reg)

best_xgb = grid_search.best_estimator_
y_pred = np.clip(best_xgb.predict(X_test_scaled), 0, 3)

In [ ]:
mse = mean_squared_error(y_test_reg, y_pred)
print("MSE:", mse)

In [ ]:
best_xgb

In [ ]:
# Quantile regression: predict 50th percentile (median)
params = {
    'objective': 'quantile',
    'alpha': 0.5,  # 0.5 for median, 0.1 for lower quantile, 0.9 for upper
    'metric': 'quantile',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'min_data_in_leaf': 20,
    'max_bin': 255,
    'verbose': -1
}

lgb_train = lgb.Dataset(X_train_scaled, y_train_reg)
lgb_val = lgb.Dataset(X_test_scaled, y_test_reg, reference=lgb_train)

model_lgb = lgb.train(
    params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_train, lgb_val]
)

# Predict
y_pred = model_lgb.predict(X_test_scaled)
# Clip predictions to CCF bounds
y_pred = np.clip(y_pred, 0, 3)

# Evaluate
print("MAE:", mean_absolute_error(y_test_reg, y_pred))
print("MSE:", mean_squared_error(y_test_reg, y_pred))


In [ ]:
# Create pools
train_pool = Pool(X_train_scaled, y_train_reg)
val_pool = Pool(X_test_scaled, y_test_reg)

# CatBoost with quantile regression
model_cb = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function='Quantile:alpha=0.5',  # median
    eval_metric='MAE',
    early_stopping_rounds=50,
    verbose=100
)

model_cb.fit(train_pool, eval_set=val_pool)

# Predict
y_pred_cb = model_cb.predict(X_test_scaled)
y_pred_cb = np.clip(y_pred_cb, 0, 3)

print("MAE:", mean_absolute_error(y_test_reg, y_pred_cb))
print("MSE:", mean_squared_error(y_test_reg, y_pred_cb))

In [ ]:
# Create and train the SVR model
svr_model = SVR(kernel='rbf', C=0.25, epsilon=0.1).fit(X_train_scaled, y_train_reg)
y_reg_pred = svr_model.predict(X_test_scaled)

In [ ]:
# Side by side
result = np.column_stack((y_reg_pred, y_test_reg))

In [ ]:
result

In [ ]:
mse = mean_squared_error(y_test_reg, y_reg_pred)
print("MSE:", mse)

In [ ]:
log_reg = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_scaled, y_train_clf)
y_clf_pred = log_reg.predict(X_test_scaled)

In [ ]:
# Side by side
result = np.column_stack((y_clf_pred, y_test_clf))

In [ ]:
result

In [ ]:
# Accuracy
accuracy = accuracy_score(y_test_clf, y_clf_pred)

# Precision, Recall, F1
precision = precision_score(y_test_clf, y_clf_pred)
recall = recall_score(y_test_clf, y_clf_pred)
f1 = f1_score(y_test_clf, y_clf_pred)

# ROC-AUC (if binary classification)
roc_auc = roc_auc_score(y_test_clf, y_clf_pred)

# Confusion Matrix
cm = confusion_matrix(y_test_clf, y_clf_pred)

# Classification Report (all metrics in one)
report = classification_report(y_test_clf, y_clf_pred)

# Print results
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("ROC AUC:", roc_auc)
print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", report)

In [ ]:
dec_tree_clf = DecisionTreeClassifier(random_state=42).fit(X_train_scaled, y_train_clf)
y_clf_pred = dec_tree_clf.predict(X_test_scaled)

In [ ]:
# Side by side
result = np.column_stack((y_clf_pred, y_test_clf))

In [ ]:
result

In [ ]:
dt = DecisionTreeClassifier(random_state=42)
param_grid = {
    'max_depth': [None, 3, 5, 7, 10, 15],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5, 10],
    'max_features': [None, 'sqrt', 'log2']
}

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=dt,
                           param_grid=param_grid,
                           cv=5,  # 5-fold cross-validation
                           scoring='roc_auc',
                           n_jobs=-1)

# Fit Grid Search
grid_search.fit(X_train_scaled, y_train_clf)


# Best hyperparameters
print("Best Hyperparameters:", grid_search.best_params_)

In [ ]:
# Evaluate on test set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_scaled)
ras = roc_auc_score(y_test_clf, y_pred)
print("Test ROC-AUC:", ras)

In [ ]:

# Accuracy
accuracy = accuracy_score(y_test_clf, y_clf_pred)

# Precision, Recall, F1
precision = precision_score(y_test_clf, y_clf_pred)
recall = recall_score(y_test_clf, y_clf_pred)
f1 = f1_score(y_test_clf, y_clf_pred)

# ROC-AUC (if binary classification)
roc_auc = roc_auc_score(y_test_clf, y_clf_pred)

# Confusion Matrix
cm = confusion_matrix(y_test_clf, y_clf_pred)

# Classification Report (all metrics in one)
report = classification_report(y_test_clf, y_clf_pred)

# Print results
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("ROC AUC:", roc_auc)
print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", report)

In [ ]:
rdm_for_clf = RandomForestClassifier(random_state=42, n_estimators=1000).fit(X_train_scaled, y_train_clf)
y_clf_pred = rdm_for_clf.predict_proba(X_test_scaled)[:,1]

In [ ]:
y_clf_pred

In [ ]:
# Side by side
result = np.column_stack((y_clf_pred, y_test_clf))

In [ ]:
result

In [ ]:
# ROC-AUC (if binary classification)
roc_auc = roc_auc_score(y_test_clf, y_clf_pred)
roc_auc

In [ ]:
# Accuracy
accuracy = accuracy_score(y_test_clf, y_clf_pred)

# Precision, Recall, F1
precision = precision_score(y_test_clf, y_clf_pred)
recall = recall_score(y_test_clf, y_clf_pred)
f1 = f1_score(y_test_clf, y_clf_pred)

# ROC-AUC (if binary classification)
roc_auc = roc_auc_score(y_test_clf, y_clf_pred)

# Confusion Matrix
cm = confusion_matrix(y_test_clf, y_clf_pred)

# Classification Report (all metrics in one)
report = classification_report(y_test_clf, y_clf_pred)

# Print results
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("ROC AUC:", roc_auc)
print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", report)

In [ ]:
pos = (y_train_clf==1).sum()
neg = (y_train_clf==0).sum()

In [ ]:
xgb_clf = XGBClassifier(scale_pos_weight=neg/pos, use_label_encoder=False, eval_metric='auc', random_state=42, n_estimators=500, max_depth=7, learning_rate = 0.1).fit(X_train, y_train_clf)
y_clf_pred = xgb_clf.predict(X_test)

In [ ]:
#y_clf_pred_xgb = xgb_clf.predict_proba(sampled_df)

In [ ]:
result = np.column_stack((y_clf_pred, y_test_clf))

In [ ]:
result

In [ ]:
X_train

In [ ]:
from xgboost import plot_importance
# --- Method 1: Using XGBoost plot_importance ---
plt.figure(figsize=(10, 8))
plot_importance(xgb_clf, importance_type='gain', max_num_features=20)
plt.title("Top 20 Feature Importances (Gain)")
plt.show()

# --- Method 2: Access feature_importances_ directly ---
# This returns normalized importance (sum=1)
importances = xgb_clf.feature_importances_
feature_names = X_train.columns

# Create a sorted bar plot
sorted_idx = importances.argsort()[::-1]
plt.figure(figsize=(12, 6))
plt.bar(range(len(importances)), importances[sorted_idx], align='center')
plt.xticks(range(len(importances)), [feature_names[i] for i in sorted_idx], rotation=90)
plt.title("XGBoost Feature Importances")
plt.tight_layout()
plt.show()

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, accuracy_score
import numpy as np

# Define the base XGBClassifier
xgb_clf = XGBClassifier(
    eval_metric='auc',   # for binary classification
    random_state=42
)

# Define hyperparameter grid
param_grid = {
    'n_estimators': [300, 500, 800],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.9, 1],
    'colsample_bytree': [0.7, 0.9, 1],
    'min_child_weight': [1, 5, 10]
}

# GridSearchCV setup
grid_search = GridSearchCV(
    estimator=xgb_clf,
    param_grid=param_grid,
    scoring='roc_auc',  # optimize for AUC
    cv=3,
    verbose=2,
    n_jobs=-1
)

# Fit GridSearchCV
grid_search.fit(X_train_scaled, y_train_clf)

# Best estimator
best_xgb_clf = grid_search.best_estimator_
print("Best parameters:", grid_search.best_params_)

# Predict on test set
y_clf_pred = best_xgb_clf.predict(X_test_scaled)
y_clf_proba = best_xgb_clf.predict_proba(X_test_scaled)[:, 1]

# Evaluate
auc = roc_auc_score(y_test_clf, y_clf_proba)
acc = accuracy_score(y_test_clf, y_clf_pred)
print("Test AUC:", auc)
print("Test Accuracy:", acc)


In [ ]:
#y_hurdle = y_clf_pred_xgb[:,1] * y_reg_pred_xgb

In [ ]:
#mean_squared_error(y_realized, y_hurdle)

In [ ]:
# Accuracy
accuracy = accuracy_score(y_test_clf, y_clf_pred)

# Precision, Recall, F1
precision = precision_score(y_test_clf, y_clf_pred)
recall = recall_score(y_test_clf, y_clf_pred)
f1 = f1_score(y_test_clf, y_clf_pred)

# ROC-AUC (if binary classification)
roc_auc = roc_auc_score(y_test_clf, y_clf_pred)

# Confusion Matrix
cm = confusion_matrix(y_test_clf, y_clf_pred)

# Classification Report (all metrics in one)
report = classification_report(y_test_clf, y_clf_pred)

# Print results
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("ROC AUC:", roc_auc)
print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", report)